<h1 align="center">🦜 LangChain — Runnables</h1>


---

> 📌 **Goal:** Understand how LangChain Runnables work and how to compose them using LCEL.

**Topics Covered:**
- What is a Runnable?
- RunnablePassthrough
- Chaining components


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")


<h1 align="center">Runnable Passthrough</h1>

> Special Runnable primitive that simply returns the input as output without modifying it.

> Like in last application when we were trying to generate Joke and then passing that to next chain to explain it .. We were not able print that Joke as well.

> Here we have solution to that - Using parallel chain after First chain generates Jokes , in parallel chain we will use one Runnable Passthrough and one Sequential Chain

```
Runnable Passthrough will give result as it is and we will get Joke as well
```
 

In [10]:
from langchain_core.runnables import RunnablePassthrough, RunnableSequence, RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt1 = PromptTemplate(
    template='Generate a joke on {topic}.',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Give me explanation for Joke : {joke}',
    input_variable=['joke']
)

parser = StrOutputParser()

firstChain = RunnableSequence(prompt1,model,parser)

parallelChain = RunnableParallel({
    "joke":RunnablePassthrough(),
    "explanation":RunnableSequence(prompt2,model,parser)
})

final_chain = RunnableSequence(firstChain, parallelChain)

jokeGenerated=final_chain.invoke({'topic':'Cricket'})

In [11]:
print(jokeGenerated['joke'])
print(jokeGenerated['explanation'])

Why did the cricket ball go to therapy?. 

Because it was feeling a little "bowled" over by its emotions.
A classic play on words. This joke is a pun, which is a type of wordplay that uses multiple meanings of a word to create humor. Let's break it down:

* "Bowled" has a double meaning here:
	+ In cricket, a bowler is a player who throws the ball to the batsman. When a batsman is "bowled," it means they are hit by the ball and their wicket is knocked over, resulting in them being dismissed.
	+ "Bowled over" is also an idiomatic expression that means being completely overwhelmed or surprised by something, often in a positive way (e.g., "I was bowled over by the news").
* In this joke, the cricket ball is feeling "a little bowled over" by its emotions, which is a clever play on words. The ball is using the cricket term "bowled" to describe its emotional state, implying that it's feeling overwhelmed or knocked off balance by its feelings.
* The humor comes from the unexpected twist on th

## Runnable Lambda

> It is a runnable primitive that allows you to apply custom Python functions within an AI pipeline

> It acts as a middleware between different AI components, enabling preprocessing transformation, API calls, filtering, and post-processing in a langChain workflow.

In [12]:
from langchain_core.runnables import RunnableLambda

prompt3 = PromptTemplate(
    template='Generate a joke on {topic}.',
    input_variables=['topic']
)

jokeChain = RunnableSequence(prompt3,model,parser)

wordCounterChain = RunnableParallel({
    "joke":RunnablePassthrough(),
    "word_count":RunnableLambda(lambda x:len(x.split()))
})

res_chain = RunnableSequence(jokeChain,wordCounterChain)

response=res_chain.invoke({'topic':'AI'})

print(response)

{'joke': 'Why did the AI program go to therapy. \n\nBecause it was struggling to process its emotions.', 'word_count': 16}
